# Soil eDNA Metabarcoding Pipeline Analysis (eKOI Database)
### A Reproducible Workflow for MinION Amplicon Sequencing (JEDI & COI)
**Project:** Genorobotics Semester Project (EPFL)

**Markers:** JEDI (~460 bp COI, arthropod-optimized) & Standard COI (~658 bp Folmer)

**Database:** eKOI PR2 — a curated eukaryotic COI database covering Metazoa, Fungi, Protists, and Plants

---

## 0. Critical Review: eKOI vs MIDORI2

### Why eKOI?
The previous analysis used MIDORI2, a general-purpose COI database that is heavily biased toward Metazoa (mostly Arthropoda and Chordata). eKOI is built on the PR2 taxonomy and includes broader eukaryotic coverage:
* **Protists & Diatoms:** eKOI includes Stramenopiles, Alveolata, and other non-metazoan eukaryotes that MIDORI2 only treated as outgroups.
* **Fungi:** Better representation of soil fungi (Basidiomycota, Ascomycota).
* **Clean taxonomy:** PR2-based 9-level taxonomy without NCBI taxon ID artifacts.
* **Smaller database (15,947 sequences):** Much faster SINTAX queries compared to MIDORI2 (9.4GB). However, this also means less species-level resolution for metazoans.

### eKOI Taxonomy Mapping (PR2 → SINTAX)
| SINTAX level | PR2 field | Example |
|---|---|---|
| `d:` Domain | Kingdom | Eukaryota |
| `k:` Kingdom | Supergroup | Obazoa, TSAR, Archaeplastida |
| `p:` Phylum | Division | Opisthokonta, Alveolata |
| `c:` Class | Subdivision | Metazoa, Fungi, Apicomplexa |
| `o:` Order | Class | Tardigrada, Nemertea |
| `f:` Family | Family | resolved from Order/Family fields |
| `g:` Genus | Genus | Milnesium |
| `s:` Species | Species | Milnesium_sp. |

### Limitations
* **15,947 sequences** vs MIDORI2's 2.2M — far fewer reference sequences, especially for metazoans.
* Species-level resolution may be lower for common arthropods/chordates.
* The PR2 taxonomy hierarchy differs from NCBI — "Phylum" in the output corresponds to PR2's Division level (e.g., Opisthokonta, Stramenopiles).

### 1. The JEDI Advantage: Optimized for Soil Arthropods
The JEDI primers target a ~460 bp region of the COI gene, specifically designed for arthropod detection in environmental samples.
* **Expected Dominance:** Arthropoda (insects, collembolans, arachnids), Annelida (earthworms), and Nematoda should dominate.
* **Potential Limitation:** The shorter fragment may reduce species-level resolution compared to full-length COI.

### 2. Standard COI in Soil: Expected Challenges
The standard Folmer COI primers (~658 bp) may underperform in soil matrices:
* **DNA Degradation:** Soil humic acids degrade DNA, favoring shorter amplicons (JEDI) over longer ones (COI).
* **Primer Bias:** Folmer primers were designed for a broader metazoan range but may miss soil-specific taxa that JEDI captures.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from pathlib import Path
from Bio import SeqIO
from matplotlib import cm

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

# Define a function to clean sample names (e.g., "Sample_barcode01" -> "01")
def clean_sample_names(columns):
    return [c.replace('Sample_barcode', '').replace('Sample_', '') for c in columns]

# Auto-detect taxonomy column prefix (SILVA_, MIDORI_, or eKOI_)
def get_tax_prefix(df):
    for col in df.columns:
        if col.startswith("SILVA_"):
            return "SILVA"
        if col.startswith("MIDORI_"):
            return "MIDORI"
        if col.startswith("eKOI_"):
            return "eKOI"
    return "SILVA"

BASE = Path("out/Soil_eDNA_JEDI_COI_14_01_26")

## Table Headers & Data Structure
Inspect the comprehensive taxonomy CSV files to verify column names and data shape.

In [ ]:
# Load both datasets
df_jedi_raw = pd.read_csv(BASE / 'taxonomy_summary/comprehensive_taxonomy_JEDI.csv')
df_coi_raw = pd.read_csv(BASE / 'taxonomy_summary/comprehensive_taxonomy_COI.csv')

prefix_jedi = get_tax_prefix(df_jedi_raw)
prefix_coi = get_tax_prefix(df_coi_raw)

for label, df, pfx in [("JEDI", df_jedi_raw, prefix_jedi), ("COI", df_coi_raw, prefix_coi)]:
    print(f"\n{'='*60}")
    print(f"  {label} ({pfx} database)  \u2014  {df.shape[0]} OTUs \u00d7 {df.shape[1]} columns")
    print(f"{'='*60}")
    
    sample_cols = [c for c in df.columns if c.startswith("Sample_")]
    taxonomy_cols = [c for c in df.columns if c.startswith(f"{pfx}_") or c.startswith("NCBI_")]
    meta_cols = [c for c in df.columns if c not in sample_cols + taxonomy_cols]
    
    print(f"\nMetadata columns:  {meta_cols}")
    print(f"Sample columns:    {sample_cols}")
    print(f"Taxonomy columns:  {taxonomy_cols}")
    print(f"\nFirst 3 rows:")
    display(df.head(3))

## Confidence Distribution Dashboard

In [ ]:
# === SINTAX Confidence Distribution Dashboard ===
ranks = ['Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']

def plot_confidence_row(df, prefix, marker_label, axes):
    """Plot confidence metrics for one marker."""
    conf_cols = [f'{prefix}_{r}_Conf' for r in ranks]
    existing = [c for c in conf_cols if c in df.columns]
    if not existing:
        for ax in axes:
            ax.text(0.5, 0.5, 'No confidence columns found.\nRegenerate with:\nbash regenerate_taxonomy.sh',
                    ha='center', va='center', transform=ax.transAxes, fontsize=10, color='red')
            ax.set_title(marker_label)
        return

    means, counts = [], []
    for r in ranks:
        col = f'{prefix}_{r}_Conf'
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors='coerce').dropna()
            means.append(vals.mean() if len(vals) > 0 else 0)
            counts.append(len(vals))
        else:
            means.append(0)
            counts.append(0)

    ax1 = axes[0]
    colors = ['#2ecc71' if m >= 0.8 else '#e74c3c' for m in means]
    bars = ax1.bar(ranks, means, color=colors, edgecolor='white')
    ax1.axhline(y=0.8, color='red', linestyle='--', alpha=0.7, label='Threshold (0.8)')
    ax1.set_ylim(0, 1.05)
    ax1.set_ylabel('Mean Confidence')
    ax1.set_title(f'{marker_label}: Mean Confidence per Rank', fontweight='bold')
    ax1.legend(fontsize=8)
    ax1.tick_params(axis='x', rotation=45)
    for bar, m, n in zip(bars, means, counts):
        if m > 0:
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{m:.2f} (n={n})', ha='center', va='bottom', fontsize=7)

    hist_rank = 'Genus' if prefix == 'SILVA' else 'Phylum'
    hist_col = f'{prefix}_{hist_rank}_Conf'
    ax2 = axes[1]
    if hist_col in df.columns:
        vals = pd.to_numeric(df[hist_col], errors='coerce').dropna()
        if len(vals) > 0:
            ax2.hist(vals, bins=20, range=(0, 1), color='steelblue', edgecolor='white', alpha=0.8)
            ax2.axvline(x=0.8, color='red', linestyle='--', linewidth=2, label='Threshold (0.8)')
            ax2.set_xlabel('Confidence')
            ax2.set_ylabel('OTU Count')
            ax2.set_title(f'{marker_label}: {hist_rank}-Level Confidence Distribution', fontweight='bold')
            ax2.legend(fontsize=8)
        else:
            ax2.text(0.5, 0.5, f'No {hist_rank} confidence values',
                    ha='center', va='center', transform=ax2.transAxes)
            ax2.set_title(f'{marker_label}: {hist_rank}-Level Confidence', fontweight='bold')
    else:
        ax2.text(0.5, 0.5, f'Column {hist_col} not found',
                ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title(f'{marker_label}: {hist_rank}-Level Confidence', fontweight='bold')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SINTAX Confidence Distribution Dashboard', fontsize=16, fontweight='bold', y=1.02)

plot_confidence_row(df_jedi_raw, prefix_jedi, f'JEDI ({prefix_jedi})', axes[0])
plot_confidence_row(df_coi_raw, prefix_coi, f'COI ({prefix_coi})', axes[1])

plt.tight_layout()
plt.show()

## Raw Read Length Distributions (Pre-Clustering)
Number of reads at each sequence length, aggregated across all barcodes for each marker.

In [ ]:
import gzip

barcode_dirs = sorted(BASE.glob("barcode*"))
marker_lengths = {"JEDI": [], "COI": []}
marker_colors_raw = {"JEDI": "#9C27B0", "COI": "#FF9800"}

for bd in barcode_dirs:
    for marker in marker_lengths:
        fq = bd / f"filtered_reads_{marker}.fastq.gz"
        if fq.exists():
            with gzip.open(str(fq), 'rt') as f:
                for i, line in enumerate(f):
                    if i % 4 == 1:
                        marker_lengths[marker].append(len(line.strip()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (marker, lengths) in enumerate(marker_lengths.items()):
    ax = axes[i]
    if lengths:
        ax.hist(lengths, bins=100, color=marker_colors_raw[marker], edgecolor="white", alpha=0.85)
        ax.axvline(np.median(lengths), color="red", ls="--", lw=1.5,
                   label=f"median={np.median(lengths):.0f}bp")
        ax.set_title(f"{marker} \u2014 {len(lengths):,} reads", fontsize=13, fontweight="bold")
        ax.set_xlabel("Read length (bp)")
        ax.set_ylabel("Number of reads")
        ax.legend()
        print(f"\u2713 {marker}: {len(lengths):,} reads, median={np.median(lengths):.0f}bp, "
              f"range={min(lengths)}-{max(lengths)}bp")
    else:
        ax.text(0.5, 0.5, f"{marker}\nNo reads found", ha='center', va='center', transform=ax.transAxes)

plt.suptitle("Raw Read Length Distributions (Soil)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Consensus OTU Sequence Length Distributions
Length distributions of consensus OTU sequences for each marker.

In [ ]:
fasta_files = {
    "JEDI": BASE / "temp_clustering/consensus_JEDI_clean.fasta",
    "COI":  BASE / "temp_clustering/consensus_COI_clean.fasta",
}
marker_colors = {"JEDI": "#9C27B0", "COI": "#FF9800"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (marker, path) in enumerate(fasta_files.items()):
    ax = axes[i]
    if path.exists():
        lengths = [len(rec.seq) for rec in SeqIO.parse(str(path), "fasta")]
        ax.hist(lengths, bins=50, color=marker_colors[marker], edgecolor="white", alpha=0.85)
        ax.axvline(np.median(lengths), color="red", ls="--", lw=1.5,
                   label=f"median={np.median(lengths):.0f}bp")
        ax.set_title(f"{marker} \u2014 {len(lengths)} OTUs", fontsize=13, fontweight="bold")
        ax.set_xlabel("Sequence length (bp)")
        ax.set_ylabel("Number of OTUs")
        ax.legend()
        print(f"\u2713 {marker}: {len(lengths)} OTUs, median={np.median(lengths):.0f}bp, "
              f"range={min(lengths)}-{max(lengths)}bp")
    else:
        ax.text(0.5, 0.5, f"{marker}\nFASTA not found", ha='center', va='center', transform=ax.transAxes)

plt.suptitle("Consensus OTU Sequence Length Distributions (Soil)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
# Part A: JEDI Marker Biodiversity Analysis (eKOI)
*Objective: Characterize the soil invertebrate community using the JEDI marker (~460 bp COI) with the eKOI PR2 reference database.*

## A.1a Broad Taxonomic Structure — Phylum
Note: In the eKOI/PR2 taxonomy, "Phylum" corresponds to the PR2 Division level (e.g., Opisthokonta, Stramenopiles, Alveolata).
* **Expected:** Opisthokonta should dominate (containing Metazoa), with possible Stramenopiles (oomycetes/diatoms) and Alveolata signals.

In [ ]:
df_jedi = df_jedi_raw.copy()
sample_cols_jedi = [c for c in df_jedi.columns if c.startswith('Sample_') and 'unclassified' not in c]
phylum_col_jedi = f'{prefix_jedi}_Phylum'

df_jedi[phylum_col_jedi] = df_jedi[phylum_col_jedi].fillna('Unassigned')
phylum_jedi = df_jedi.groupby(phylum_col_jedi)[sample_cols_jedi].sum()

phylum_jedi['Total'] = phylum_jedi.sum(axis=1)
phylum_jedi = phylum_jedi.sort_values('Total', ascending=False)
top_phyla = phylum_jedi.head(10).index

plot_data = phylum_jedi.loc[top_phyla].drop(columns='Total')
others = phylum_jedi.loc[~phylum_jedi.index.isin(top_phyla)].drop(columns='Total').sum()
plot_data.loc['Others'] = others

fig, ax = plt.subplots(figsize=(12, 6))
plot_data.columns = clean_sample_names(plot_data.columns)
plot_data.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, cmap='tab20')

ax.set_title(f'Soil Community Composition \u2014 Phylum (JEDI ~460bp, {prefix_jedi})', fontweight='bold')
ax.set_xlabel('Sample ID')
ax.set_ylabel('Relative Abundance')
ax.legend(title='Phylum', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

## A.1b Class-Level Breakdown (JEDI)
In eKOI/PR2 taxonomy, "Class" corresponds to the Subdivision level (e.g., Metazoa, Fungi, Apicomplexa).
* **Metazoa** should dominate as the primary soil eukaryote signal.
* Look for **Fungi** signals that MIDORI2 would have missed.

In [ ]:
class_col_jedi = f'{prefix_jedi}_Class'
df_jedi[class_col_jedi] = df_jedi[class_col_jedi].fillna('Unassigned')
class_jedi = df_jedi.groupby(class_col_jedi)[sample_cols_jedi].sum()
class_jedi = class_jedi.drop('Unassigned', errors='ignore')

class_jedi['Total'] = class_jedi.sum(axis=1)
class_jedi = class_jedi.sort_values('Total', ascending=False)
top_classes = class_jedi.head(15).index

plot_cls = class_jedi.loc[top_classes].drop(columns='Total')
others_cls = class_jedi.loc[~class_jedi.index.isin(top_classes)].drop(columns='Total').sum()
plot_cls.loc['Others'] = others_cls

num_colors = len(plot_cls)
colors = cm.tab20(np.linspace(0, 1, num_colors))
custom_colors = []
for i, cls_name in enumerate(plot_cls.index):
    if cls_name == 'Unassigned':
        custom_colors.append('#D3D3D3')
    elif cls_name == 'Others':
        custom_colors.append('#696969')
    else:
        custom_colors.append(colors[i])

fig, ax = plt.subplots(figsize=(12, 7))
plot_cls.columns = clean_sample_names(plot_cls.columns)
plot_cls = plot_cls.div(plot_cls.sum(axis=0), axis=1) * 100  # normalize to %
plot_cls = plot_cls.fillna(0)
plot_cls.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, color=custom_colors)

ax.set_title(f'Class-Level Composition (JEDI, {prefix_jedi})', fontweight='bold')
ax.set_ylabel('Relative Abundance (%)')
handles, labels = ax.get_legend_handles_labels()
ax.legend(reversed(handles), reversed(labels), bbox_to_anchor=(1.02, 1), loc='upper left', title='Class')

# Mark samples with no assigned taxa
for _idx, _sample in enumerate(plot_cls.columns):
    if plot_cls[_sample].sum() == 0:
        ax.text(_idx, 2, 'ND', ha='center', va='bottom',
                fontsize=9, color='gray', fontstyle='italic')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## A.2 Order-Level Breakdown (JEDI)
In eKOI/PR2, "Order" maps to the PR2 Class level (e.g., Tardigrada, Insecta, Arachnida for Metazoa).

In [ ]:
order_col_jedi = f'{prefix_jedi}_Order'
df_jedi[order_col_jedi] = df_jedi[order_col_jedi].fillna('Unassigned')
order_jedi = df_jedi.groupby(order_col_jedi)[sample_cols_jedi].sum()
order_jedi = order_jedi.drop('Unassigned', errors='ignore')

order_jedi['Total'] = order_jedi.sum(axis=1)
order_jedi = order_jedi.sort_values('Total', ascending=False)
top_orders = order_jedi.head(15).index

plot_ord = order_jedi.loc[top_orders].drop(columns='Total')
others_ord = order_jedi.loc[~order_jedi.index.isin(top_orders)].drop(columns='Total').sum()
plot_ord.loc['Others'] = others_ord

fig, ax = plt.subplots(figsize=(12, 7))
plot_ord.columns = clean_sample_names(plot_ord.columns)
plot_ord = plot_ord.div(plot_ord.sum(axis=0), axis=1) * 100  # normalize to %
plot_ord = plot_ord.fillna(0)
plot_ord.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, cmap='tab20')

ax.set_title(f'Order-Level Composition (JEDI, {prefix_jedi})', fontweight='bold')
ax.set_ylabel('Relative Abundance (%)')
ax.legend(title='Order', bbox_to_anchor=(1.02, 1), loc='upper left')

# Mark samples with no assigned taxa
for _idx, _sample in enumerate(plot_ord.columns):
    if plot_ord[_sample].sum() == 0:
        ax.text(_idx, 2, 'ND', ha='center', va='bottom',
                fontsize=9, color='gray', fontstyle='italic')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## A.3 Genus-Level Top 20 (JEDI)
Top genera detected by the JEDI marker with eKOI taxonomy.

In [ ]:
genus_col_jedi = f'{prefix_jedi}_Genus'
df_jedi[genus_col_jedi] = df_jedi[genus_col_jedi].fillna('Unassigned')
genus_jedi = df_jedi.groupby(genus_col_jedi)[sample_cols_jedi].sum()
genus_jedi = genus_jedi.drop('Unassigned', errors='ignore')
genus_jedi['Total'] = genus_jedi.sum(axis=1)
genus_jedi = genus_jedi.sort_values('Total', ascending=False)

top_genera = genus_jedi.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x=top_genera['Total'], y=top_genera.index, palette='viridis', ax=ax)
ax.set_title(f'Top 20 Genera (JEDI, {prefix_jedi})', fontweight='bold')
ax.set_xlabel('Total Abundance')
ax.set_ylabel('Genus')


# Annotate bars with mean confidence
_genus_conf_col = f'{prefix_jedi}_Genus_Conf'
if _genus_conf_col in df_jedi.columns:
    for _i, _genus_name in enumerate(top_genera.index):
        _mask = df_jedi[genus_col_jedi] == _genus_name
        _cvals = pd.to_numeric(df_jedi.loc[_mask, _genus_conf_col], errors='coerce').dropna()
        if len(_cvals) > 0:
            _mc = _cvals.mean()
            ax.text(top_genera['Total'].iloc[_i], _i, f' ({_mc:.2f})',
                    va='center', ha='left', fontsize=8, color='darkred', fontweight='bold')
plt.tight_layout()
plt.show()

## A.4 Taxonomic Resolution Decay (JEDI)
Percentage of OTUs that remain "Unassigned" at each taxonomic rank.

In [ ]:
ranks = ['Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']
levels_jedi = [f'{prefix_jedi}_{r}' for r in ranks]
unassigned_counts_jedi = []

for level in levels_jedi:
    n_unassigned = df_jedi[level].isin(['Unassigned', '', np.nan]).sum()
    pct = (n_unassigned / len(df_jedi)) * 100
    unassigned_counts_jedi.append(pct)

plt.figure(figsize=(8, 5))
sns.barplot(x=ranks, y=unassigned_counts_jedi, color="salmon")
plt.title(f'Taxonomic Resolution Decay (JEDI, {prefix_jedi})', fontweight='bold')
plt.ylabel('% of OTUs Unassigned')
plt.xticks(rotation=45)
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

---
# Part B: COI Marker Biodiversity Analysis (Standard Folmer ~658 bp, eKOI)
*Objective: Characterize the soil community using the standard COI marker with eKOI taxonomy and compare with JEDI.*

## B.1a Broad Taxonomic Structure (COI)

In [ ]:
df_coi = df_coi_raw.copy()
sample_cols_coi = [c for c in df_coi.columns if c.startswith('Sample_') and 'unclassified' not in c]
phylum_col_coi = f'{prefix_coi}_Phylum'

df_coi[phylum_col_coi] = df_coi[phylum_col_coi].fillna('Unassigned')
phylum_coi = df_coi.groupby(phylum_col_coi)[sample_cols_coi].sum()

phylum_coi['Total'] = phylum_coi.sum(axis=1)
phylum_coi = phylum_coi.sort_values('Total', ascending=False)
top_phyla_coi = phylum_coi.head(10).index

plot_data_coi = phylum_coi.loc[top_phyla_coi].drop(columns='Total')
others_coi = phylum_coi.loc[~phylum_coi.index.isin(top_phyla_coi)].drop(columns='Total').sum()
plot_data_coi.loc['Others'] = others_coi

fig, ax = plt.subplots(figsize=(12, 6))
plot_data_coi.columns = clean_sample_names(plot_data_coi.columns)
plot_data_coi.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, cmap='tab20')

ax.set_title(f'Soil Community Composition \u2014 Phylum (COI ~658bp, {prefix_coi})', fontweight='bold')
ax.set_xlabel('Sample ID')
ax.set_ylabel('Relative Abundance')
ax.legend(title='Phylum', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

## B.1b Class-Level Breakdown (COI)

In [ ]:
class_col_coi = f'{prefix_coi}_Class'
df_coi[class_col_coi] = df_coi[class_col_coi].fillna('Unassigned')
class_coi = df_coi.groupby(class_col_coi)[sample_cols_coi].sum()
class_coi = class_coi.drop('Unassigned', errors='ignore')

class_coi['Total'] = class_coi.sum(axis=1)
class_coi = class_coi.sort_values('Total', ascending=False)
top_classes_coi = class_coi.head(15).index

plot_cls_coi = class_coi.loc[top_classes_coi].drop(columns='Total')
others_cls_coi = class_coi.loc[~class_coi.index.isin(top_classes_coi)].drop(columns='Total').sum()
plot_cls_coi.loc['Others'] = others_cls_coi

num_colors_coi = len(plot_cls_coi)
colors_coi = cm.tab20(np.linspace(0, 1, num_colors_coi))
custom_colors_coi = []
for i, cls_name in enumerate(plot_cls_coi.index):
    if cls_name == 'Unassigned':
        custom_colors_coi.append('#D3D3D3')
    elif cls_name == 'Others':
        custom_colors_coi.append('#696969')
    else:
        custom_colors_coi.append(colors_coi[i])

fig, ax = plt.subplots(figsize=(12, 7))
plot_cls_coi.columns = clean_sample_names(plot_cls_coi.columns)
plot_cls_coi = plot_cls_coi.div(plot_cls_coi.sum(axis=0), axis=1) * 100  # normalize to %
plot_cls_coi = plot_cls_coi.fillna(0)
plot_cls_coi.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, color=custom_colors_coi)

ax.set_title(f'Class-Level Composition (COI, {prefix_coi})', fontweight='bold')
ax.set_ylabel('Relative Abundance (%)')
handles, labels = ax.get_legend_handles_labels()
ax.legend(reversed(handles), reversed(labels), bbox_to_anchor=(1.02, 1), loc='upper left', title='Class')

# Mark samples with no assigned taxa
for _idx, _sample in enumerate(plot_cls_coi.columns):
    if plot_cls_coi[_sample].sum() == 0:
        ax.text(_idx, 2, 'ND', ha='center', va='bottom',
                fontsize=9, color='gray', fontstyle='italic')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## B.2 Order-Level Breakdown (COI)

In [ ]:
order_col_coi = f'{prefix_coi}_Order'
df_coi[order_col_coi] = df_coi[order_col_coi].fillna('Unassigned')
order_coi = df_coi.groupby(order_col_coi)[sample_cols_coi].sum()
order_coi = order_coi.drop('Unassigned', errors='ignore')

order_coi['Total'] = order_coi.sum(axis=1)
order_coi = order_coi.sort_values('Total', ascending=False)
top_orders_coi = order_coi.head(15).index

plot_ord_coi = order_coi.loc[top_orders_coi].drop(columns='Total')
others_ord_coi = order_coi.loc[~order_coi.index.isin(top_orders_coi)].drop(columns='Total').sum()
plot_ord_coi.loc['Others'] = others_ord_coi

fig, ax = plt.subplots(figsize=(12, 7))
plot_ord_coi.columns = clean_sample_names(plot_ord_coi.columns)
plot_ord_coi = plot_ord_coi.div(plot_ord_coi.sum(axis=0), axis=1) * 100  # normalize to %
plot_ord_coi = plot_ord_coi.fillna(0)
plot_ord_coi.T.plot(kind='bar', stacked=True, ax=ax, width=0.85, cmap='tab20')

ax.set_title(f'Order-Level Composition (COI, {prefix_coi})', fontweight='bold')
ax.set_ylabel('Relative Abundance (%)')
ax.legend(title='Order', bbox_to_anchor=(1.02, 1), loc='upper left')

# Mark samples with no assigned taxa
for _idx, _sample in enumerate(plot_ord_coi.columns):
    if plot_ord_coi[_sample].sum() == 0:
        ax.text(_idx, 2, 'ND', ha='center', va='bottom',
                fontsize=9, color='gray', fontstyle='italic')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## B.3 Genus-Level Top 20 (COI)

In [ ]:
genus_col_coi = f'{prefix_coi}_Genus'
df_coi[genus_col_coi] = df_coi[genus_col_coi].fillna('Unassigned')
genus_coi = df_coi.groupby(genus_col_coi)[sample_cols_coi].sum()
genus_coi = genus_coi.drop('Unassigned', errors='ignore')
genus_coi['Total'] = genus_coi.sum(axis=1)
genus_coi = genus_coi.sort_values('Total', ascending=False)

top_genera_coi = genus_coi.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x=top_genera_coi['Total'], y=top_genera_coi.index, palette='viridis', ax=ax)
ax.set_title(f'Top 20 Genera (COI, {prefix_coi})', fontweight='bold')
ax.set_xlabel('Total Abundance')
ax.set_ylabel('Genus')


# Annotate bars with mean confidence
_genus_conf_col = f'{prefix_coi}_Genus_Conf'
if _genus_conf_col in df_coi.columns:
    for _i, _genus_name in enumerate(top_genera_coi.index):
        _mask = df_coi[genus_col_coi] == _genus_name
        _cvals = pd.to_numeric(df_coi.loc[_mask, _genus_conf_col], errors='coerce').dropna()
        if len(_cvals) > 0:
            _mc = _cvals.mean()
            ax.text(top_genera_coi['Total'].iloc[_i], _i, f' ({_mc:.2f})',
                    va='center', ha='left', fontsize=8, color='darkred', fontweight='bold')
plt.tight_layout()
plt.show()

## B.4 Taxonomic Resolution Comparison: JEDI vs COI (eKOI)

In [ ]:
ranks = ['Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']

levels_jedi = [f'{prefix_jedi}_{r}' for r in ranks]
unassigned_jedi = []
for level in levels_jedi:
    n_unassigned = df_jedi[level].isin(['Unassigned', '', np.nan]).sum()
    pct = (n_unassigned / len(df_jedi)) * 100
    unassigned_jedi.append(pct)

levels_coi = [f'{prefix_coi}_{r}' for r in ranks]
unassigned_coi = []
for level in levels_coi:
    n_unassigned = df_coi[level].isin(['Unassigned', '', np.nan]).sum()
    pct = (n_unassigned / len(df_coi)) * 100
    unassigned_coi.append(pct)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(ranks))
width = 0.35

bars1 = ax.bar(x - width/2, unassigned_jedi, width, label=f'JEDI ({prefix_jedi})', color='salmon', alpha=0.8)
bars2 = ax.bar(x + width/2, unassigned_coi, width, label=f'COI ({prefix_coi})', color='steelblue', alpha=0.8)

ax.set_title('Taxonomic Resolution Decay: JEDI vs COI (Soil, eKOI)', fontweight='bold')
ax.set_ylabel('% of OTUs Unassigned')
ax.set_xticks(x)
ax.set_xticklabels(ranks, rotation=45)
ax.set_ylim(0, 100)
ax.legend()

plt.tight_layout()
plt.show()

## Forensics: BLAST Identification (JEDI + COI)
BLAST validation of the top OTUs for both markers provides ground-truth species identification beyond what SINTAX can offer with the eKOI database.

In [ ]:
def parse_blast_file(filepath):
    """Parses the custom text output from script 6_blast_top_otus.py"""
    data = []
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()
        start_reading = False
        for line in lines:
            if line.startswith('---'):
                start_reading = True
                continue
            if not start_reading or not line.strip(): continue
            parts = line.split('|')
            if len(parts) >= 3:
                species = parts[2].strip()
                if "Uncultured" in species: species = "Uncultured organism"
                try:
                    reads = float(parts[1].strip())
                    data.append({'Species': species, 'Abundance': reads})
                except: continue
    except FileNotFoundError:
        print("BLAST file not found.")
        return pd.DataFrame()
    return pd.DataFrame(data)

for marker in ['JEDI', 'COI']:
    df_blast = parse_blast_file(f'out/Soil_eDNA_JEDI_COI_14_01_26/blast_results/blast_top10_{marker}.txt')
    if not df_blast.empty:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=df_blast, y='Species', x='Abundance', hue='Species', palette='magma', legend=False)
        plt.title(f'True Identity of Top {marker} OTUs (BLAST)', fontweight='bold')
        plt.xlabel('Relative Abundance')
        plt.tight_layout()
        plt.show()
    else:
        print(f"No valid BLAST data found for {marker}.")

---
## Part C: eKOI Non-Metazoan Detections
One of the main advantages of eKOI over MIDORI2 is better coverage of non-metazoan eukaryotes. Let's examine what we find beyond Metazoa.

In [ ]:
# Combine JEDI + COI to look at non-metazoan detections
for label, df, pfx, sample_cols in [("JEDI", df_jedi, prefix_jedi, sample_cols_jedi), 
                                      ("COI", df_coi, prefix_coi, sample_cols_coi)]:
    class_col = f'{pfx}_Class'
    phylum_col = f'{pfx}_Phylum'
    genus_col = f'{pfx}_Genus'
    
    # Non-metazoan = Class is not Metazoa and not empty/Unassigned
    non_metazoan = df[
        (~df[class_col].isin(['Metazoa', 'Unassigned', '', np.nan])) & 
        (df[class_col].notna())
    ]
    
    if len(non_metazoan) > 0:
        print(f"\n{'='*60}")
        print(f"  {label}: {len(non_metazoan)} non-Metazoan OTUs ({100*len(non_metazoan)/len(df):.1f}%)")
        print(f"{'='*60}")
        
        # Group by Class (Subdivision)
        non_meta_class = non_metazoan.groupby(class_col)[sample_cols].sum()
        non_meta_class['Total'] = non_meta_class.sum(axis=1)
        non_meta_class = non_meta_class.sort_values('Total', ascending=False)
        print(f"\nNon-Metazoan Classes:")
        for cls, row in non_meta_class.head(10).iterrows():
            print(f"  {cls}: {row['Total']:.4f}")
        
        # Top non-metazoan genera
        non_meta_genus = non_metazoan.groupby(genus_col)[sample_cols].sum()
        non_meta_genus['Total'] = non_meta_genus.sum(axis=1)
        non_meta_genus = non_meta_genus.sort_values('Total', ascending=False)
        print(f"\nTop 10 non-Metazoan Genera:")
        for gen, row in non_meta_genus.head(10).iterrows():
            print(f"  {gen}: {row['Total']:.4f}")
    else:
        print(f"\n{label}: No non-Metazoan OTUs detected")